# W3Q4 — Cumulative Conversion Curves (2022 Cohorts)
**Question:** For users who signed up in 2022, how quickly does conversion accumulate over time? Are some cohorts faster than others?

**Audience:** Marketing & Leadership  
**Data source:** `ANALYTICS.MARTS.MART_SUBSCRIPTION_FUNNEL`  
**SQL:** `sql/W3Q4_cumulative_conversion_curves.sql`

---

> ⚠️ **Data note (DN-001):** Signup data capped at November 2022.
> Recent cohorts (Sep–Nov 2022) have had less time to convert — their curves will be truncated.
> Interpret with caution and focus on Jan–Aug 2022 cohorts for completed arcs.

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from src.connection import query

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130})
PALETTE = sns.color_palette('muted', 8)

## Load Data

In [ ]:
with open('../sql/W3Q4_cumulative_conversion_curves.sql') as f:
    sql = f.read()

df = query(sql)
df['cohort_month'] = pd.to_datetime(df['cohort_month'])
print(f"Rows: {len(df):,}  |  Cohorts: {df['cohort_month'].nunique()}")
df.head(3)

## Compute Cumulative Conversion at Fixed Time Thresholds

In [ ]:
thresholds = [7, 14, 30, 60, 90, 180, 365]

def cumulative_rate(df_cohort, col_days, thresholds):
    n = len(df_cohort)
    return {t: (df_cohort[col_days].fillna(9999) <= t).sum() / n * 100 for t in thresholds}

results = []
for cohort, grp in df.groupby('cohort_month'):
    row = {'cohort_month': cohort, 'n': len(grp)}
    for col, stage in [('days_signup_to_trial', 'trial'), ('days_signup_to_3m', '3m'), ('days_signup_to_12m', '12m')]:
        rates = cumulative_rate(grp, col, thresholds)
        for t, rate in rates.items():
            row[f'{stage}_{t}d'] = round(rate, 1)
    results.append(row)

curves = pd.DataFrame(results)
display(curves.head())

## Cumulative Conversion Curves — Trial

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
cutoff = pd.Timestamp('2022-09-01')

for _, row in curves.iterrows():
    y = [row[f'trial_{t}d'] for t in thresholds]
    color = '#d9534f' if row['cohort_month'] >= cutoff else PALETTE[0]
    alpha = 0.5 if row['cohort_month'] >= cutoff else 0.8
    ax.plot(thresholds, y, color=color, alpha=alpha, linewidth=1.5,
            label=row['cohort_month'].strftime('%b %Y'))

ax.set_title('Cumulative Trial Conversion — 2022 Cohorts\n(Red = recent cohorts with insufficient observation time)', fontsize=13, pad=12)
ax.set_xlabel('Days Since Signup')
ax.set_ylabel('Cumulative % Converted to Trial')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.set_xticks(thresholds)
sns.despine()
plt.tight_layout()
plt.savefig('../output/W3Q4_cumulative_trial_conversion.png', bbox_inches='tight')
plt.show()

## Cumulative Conversion Curves — 3m & 12m

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (stage, label) in zip(axes, [('3m', '3m Subscription'), ('12m', '12m Subscription')]):
    for _, row in curves.iterrows():
        y = [row[f'{stage}_{t}d'] for t in thresholds]
        color = '#d9534f' if row['cohort_month'] >= cutoff else PALETTE[2]
        alpha = 0.5 if row['cohort_month'] >= cutoff else 0.8
        ax.plot(thresholds, y, color=color, alpha=alpha, linewidth=1.5)
    ax.set_title(f'Cumulative {label} Conversion', fontsize=12)
    ax.set_xlabel('Days Since Signup')
    ax.set_ylabel('Cumulative % Converted')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    ax.set_xticks(thresholds)
    sns.despine(ax=ax)

fig.suptitle('Cumulative Conversion Curves — 2022 Cohorts (Red = truncated cohorts)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../output/W3Q4_cumulative_paid_conversion.png', bbox_inches='tight')
plt.show()

## Findings

- **Trial conversion speed:** ...
- **3m conversion speed:** ...
- **12m conversion speed:** ...
- **Cohort variation (fast vs slow cohorts):** ...
- **Recent cohorts (Sep–Nov 2022):** ...
- **Recommendation:** ...